# Bias mitigiation using sub-sampled synthesized data
---
The goal of this notebook is to use different tools to check whether the automatic bias detected is also automatically mitigatable. Intermediate results are displayed directly here, for summarized data, have a look at the thesis.

In [ ]:
import sys

sys.path.append("../src")  # go to parent dir
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from data_utils import split_data, prepare_data_fair_learning
from models_utils import evaluate_model, evaluate_model, train_and_evaluate_pipeline, calculate_utility_metrics
from fairness_utils import (
    search_bias,
    evaluate_fairness_score,
    explain_bias,
    encode_protected_attributes,
    evaluate_fairness,
    convert_to_standardDataset,
    get_fair_learning_scoring,
    train_and_evaluate_fairness_pipeline
)
from sklearn import clone
from aif360.algorithms.preprocessing import Reweighing, DisparateImpactRemover, LFR
from sklearn.model_selection import GridSearchCV
from aif360.sklearn.preprocessing import LearnedFairRepresentations

warnings.filterwarnings("ignore")
random_state = 12041500

## Load data
We start by loading the respective data from the `./data` directory.

In [34]:
def load_data():
    df_train = pd.read_json("../data/synthetic_data_CTGANSynthesizer.json").drop(columns=["fnlwgt"])
    df_test = pd.read_json("../data/testset.json").drop(columns=["fnlwgt"])
    
    return df_train, df_test

ratio_features = ["age", "capital-gain", "capital-loss", "hours-per-week"]
ordinal_features = ["education-num"]
nominal_features = ["workclass", "marital-status", "occupation", "relationship", "race", "sex"]
target = "income"

In [35]:
df_train, df_test = load_data()

### Subsample

In [36]:
def subsample(df):
    # Apply biasing rules
    data_less_equal_50k_Husband = df[(df_train["income"] == 0) & (df_train["relationship"] == "Husband")]
    data_greater_50k_Husband = df[(df_train["income"] == 1) & (df_train["relationship"] == "Husband")]
    data_unequal_Husband = df[df_train["relationship"] != "Husband"]

    # Undersample the "less equal 50k Husband" group
    data_less_equal_50k_Husband_undersampled = resample(
        data_less_equal_50k_Husband, replace=False, 
        n_samples=int(len(data_less_equal_50k_Husband) * 0.20), 
        random_state=random_state
    )

    # Create biased dataset
    biased_data = pd.concat([data_unequal_Husband, data_greater_50k_Husband, data_less_equal_50k_Husband_undersampled])

    # Shuffle and remove duplicates
    df = biased_data.sample(frac=1, random_state=random_state).reset_index(drop=True)
    df.drop_duplicates(inplace=True)
    return df

df_train = subsample(df_train)

#### Visualization

In [37]:
from dython.nominal import associations
import matplotlib.pyplot as plt
import seaborn as sns

def plot_correlation_compare():
    df_train_non_sampled, _ = load_data()
    corr1 = associations(df_train_non_sampled, compute_only=True)
    corr2 = associations(df_train, compute_only=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    sns.heatmap(corr1['corr'], ax=ax1, cmap="Reds", annot=True, cbar=False)
    sns.heatmap(corr2['corr'], ax=ax2, cmap="Reds", annot=True, cbar=False)
    ax1.set_title('Correlation for synthetic data')
    ax2.set_title('Correlation for skewed data')
    plt.tight_layout()
    plt.show()

plot_correlation_compare()

## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [38]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [39]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.776
Precision       : 0.720
Recall          : 0.779
F1              : 0.733


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [40]:
X_train, y_train = split_data(df_train, target, drop_na=True)
X_test, y_test = split_data(df_test, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])
probs_test = pd.Series(model.predict_proba(X_test)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [11]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [0, 1, 3, 4, 5, 6]}, 136.2551)


Check it for test data. 💡 Just update the ramaining privileged subset used to get other results

In [ ]:
privileged_subset_test, _ = search_bias(X_test, y_test, probs_test, 1, penalty=1)
print(privileged_subset_test)

({'relationship': ['Husband', 'Other-relative', 'Own-child'], 'capital-loss': [0, 1258, 1411, 1579, 1628, 1672, 1740, 2002, 2042, 2051, 2057, 2149, 2179, 2377, 2457, 2467, 2603], 'capital-gain': [0, 1173, 1409, 1797, 2228, 2290, 2407, 2414, 2580, 2635, 2653, 2829, 2885, 3137, 3411, 3432, 3456, 3464, 3471, 3781, 3818, 3908, 3942, 4064, 4508, 5013, 6767, 10566, 41310]}, 1136.1445)


In [41]:
privileged_subset = ({'relationship': ['Husband']}, 0)

In [42]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['relationship']

         Group Distance  Proportion  Counts   P-Value
     Own-child   -0.226    0.121204    3788  0.00e+00
       Husband    0.556    0.222155    6943  0.00e+00
 Not-in-family   -0.167    0.377724   11805 4.94e-324
     Unmarried   -0.209    0.134387    4200 4.06e-306
Other-relative   -0.220    0.045084    1409 9.00e-120
          Wife    0.050    0.099446    3108  2.05e-10

Weighted Mean Statistical Distance: 0.25698147498046253


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [43]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 7090, we observe 0.7914 as the average probability of earning >50k, but our model predicts 0.2365


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [44]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

1398 Na rows removed!
202 Na rows removed!


Lastly, we can compute the respective metrics

In [45]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   -0.885
average_odds_difference         -0.706
equal_opportunity_difference    -0.692
disparate_impact                0.039
theil_index                     0.071


Using the test set:

In [46]:
_ = evaluate_fairness(
    df_test_bias[target],
    model.predict(df_test_bias.drop(columns=[target])),
    list(privileged_subset[0].keys()),
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   -0.833
average_odds_difference         -0.772
equal_opportunity_difference    -0.791
disparate_impact                0.027
theil_index                     0.100


## Mitigation
Next, we use different mitigation techniques to solve the problem. More information can be found within the thesis itself.

### Reweighting
We start by creating the privileged and unprivileged groups.

In [47]:
# create (un)privileged groups
privileged_groups = [{key: 1 for key in list(privileged_subset[0].keys())}]
unprivileged_groups = [{key: 0 for key in list(privileged_subset[0].keys())}]

Next, we convert our datasets to StandardDatasets implemented through AIF360, such that their provided implementations work with our data.

In [48]:
# convert standard dataset (sd)
sd_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
sd_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

Lastly, we fit and transform the dataset using the Reweighing strategy.

In [49]:
RW = Reweighing(unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
sd_reweigh = RW.fit_transform(sd_train)

#### Baseline
We again establish a baseline, such that comparing results is easier

In [50]:
model = clone(clf)
model.fit(sd_train.features, sd_train.labels.ravel())

_ = evaluate_model(model, sd_test.features, sd_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.767
Precision       : 0.714
Recall          : 0.773
F1              : 0.725


In [51]:
_ = evaluate_fairness(df_train_bias[target], model.predict(sd_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   -0.907
average_odds_difference         -0.754
equal_opportunity_difference    -0.738
disparate_impact                0.033
theil_index                     0.072


#### Mitigated Model
Next, we apply the transform dataset, to see whether it has an positive effect.

In [52]:
model = clone(clf)
model.fit(sd_reweigh.features, sd_reweigh.labels.ravel(), sample_weight=sd_reweigh.instance_weights)

_ = evaluate_model(model, sd_test.features, sd_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.828
Precision       : 0.783
Recall          : 0.704
F1              : 0.729


In [53]:
_ = evaluate_fairness(df_train_bias[target], model.predict(sd_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   -0.210
average_odds_difference         0.085
equal_opportunity_difference    0.168
disparate_impact                0.329
theil_index                     0.170


### Fair Learning
Next, we consider Fair Learning, for this we reload the data and prepare it for the fair learning technique.

In [55]:
df_train, df_test = load_data()
df_train = subsample(df_train)

In [56]:
df_train_bias = encode_protected_attributes(df_train.dropna(), list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test.dropna(), list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

X_train, y_train, X_test, y_test = prepare_data_fair_learning(df_train_bias, df_test_bias, nominal_features, target)

In [57]:
X_train['capital-gain'] = X_train.index
X_test['capital-gain'] = X_test.index

In [58]:
max_delta = get_fair_learning_scoring(list(privileged_subset[0].keys()))

Next, we prepare the algorithm and the params.

In [59]:
lfr = LearnedFairRepresentations(
    list(privileged_subset[0].keys()),
    n_prototypes=20,
    max_iter=50,
    random_state=random_state,
)

In [30]:
params = {
    "reconstruct_weight": [1e-2, 1e-3, 1e-4],
    "target_weight": [100, 1000],
    "fairness_weight": [0, 100, 1000],
}

Lastly, we perform grid search to get to the best results.

In [ ]:
grid = GridSearchCV(lfr, params, scoring=max_delta, cv=3, n_jobs=-1).fit(
    X_train, y_train, priv_group=(1,) * len(list(privileged_subset[0].keys()))
)
res = pd.DataFrame(grid.cv_results_)

#### Baseline
Again, we start by implementing the baseline.

In [ ]:
model = clone(clf)
model.fit(X_train, y_train)

_ = evaluate_model(model, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.783
Precision      0.725
Recall         0.779
F1             0.739


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(X_train), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   -0.859
average_odds_difference         -0.687
equal_opportunity_difference    -0.694
disparate_impact                0.038
theil_index                     0.075


#### Using Grid itself
Next, we simply use the trained grid-search object to perform the predictions.

In [ ]:
_ = evaluate_model(grid, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.776
Precision      0.696
Recall         0.583
F1             0.589


In [ ]:
_ = evaluate_fairness(df_train_bias[target], grid.predict(X_train), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   -0.426
average_odds_difference         -0.210
equal_opportunity_difference    -0.225
disparate_impact                0.163
theil_index                     0.142


#### Transforming data
Next, we use the grid to transform the training data into the respective object and use its results.

In [ ]:
model = clone(clf)
model.fit(grid.transform(X_train), y_train)

_ = evaluate_model(model, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.759
Precision      0.380
Recall         0.500
F1             0.432


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(X_test), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   0.000
average_odds_difference         0.000
equal_opportunity_difference    0.000
disparate_impact                0.000
theil_index                     0.276


##### Also Transforming test data

In [ ]:
_ = evaluate_model(model, grid.transform(X_test), y_test, verbose=True)

Metric         Value               
Accuracy       0.776
Precision      0.696
Recall         0.583
F1             0.590


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(grid.transform(X_train)), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   -0.495
average_odds_difference         -0.285
equal_opportunity_difference    -0.312
disparate_impact                0.139
theil_index                     0.130


Multiple Runs cannot be done, because of the dataset already needs to be one hot encoded for the mitigation, making the `search_bias` function unnecessary and returning not viable resolutions.

### Fair Learning (AIF360)
Next, we look at the Fair Learning implementation by AIF360.

In [60]:
df_train, df_test = load_data()
df_train = subsample(df_train)

In [61]:
ds_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
ds_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

In [62]:
privileged_groups = [{key: 1 for key in list(privileged_subset[0].keys())}]
unprivileged_groups = [{key: 0 for key in list(privileged_subset[0].keys())}]

In [63]:
TR = LFR(unprivileged_groups=unprivileged_groups,
         privileged_groups=privileged_groups,
         k=10, Ax=0.01, Ay=1000, Az=0,
         verbose=1,
         seed=random_state
)

TR = TR.fit(ds_train, maxiter=5000, maxfun=1000)

step: 0, loss: 4463.16265723971, L_x: 384912.80286332034,  L_y: 0.6140346286065067,  L_z: 0.0052036359449185
step: 250, loss: 4463.162655303932, L_x: 384912.8028633387,  L_y: 0.6140346266705456,  L_z: 0.005203636870659628
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =          450     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  4.46316D+03    |proj g|=  3.60946D+01
step: 500, loss: 4409.6284288845345, L_x: 384912.43828122143,  L_y: 0.5605040460723204,  L_z: 0.005517235969012113
step: 750, loss: 4409.628431799466, L_x: 384912.43828123726,  L_y: 0.5605040489870932,  L_z: 0.00551723520230636

At iterate    1    f=  4.40963D+03    |proj g|=  1.56100D+01
step: 1000, loss: 4377.905517690482, L_x: 384910.71599650715,  L_y: 0.5287983577254105,  L_z: 0.010282045552248329
step: 1250, loss: 4377.90551806545, L_x: 384910.7159965432,  L_y: 0.5287983581000185,  L_z: 0.010282045242572879

At iterate    2    f=  4.37791

Transform training data and align features

In [64]:
ds_train_lfr = TR.transform(ds_train)

#### Baseline
We again start by establishing a baseline.

In [65]:
model = clone(clf)
model.fit(ds_train.features, ds_train.labels.ravel())

_ = evaluate_model(model, ds_test.features, ds_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.767
Precision       : 0.714
Recall          : 0.773
F1              : 0.725


In [66]:
_ = evaluate_fairness(df_train_bias[target], model.predict(ds_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   -0.907
average_odds_difference         -0.754
equal_opportunity_difference    -0.738
disparate_impact                0.033
theil_index                     0.072


#### Mitigation
Since the tool does simply convert all labels to 0, its easy to see why those results are not promising and should not be further considered.

In [75]:
model = clone(clf)
model.fit(ds_train_lfr.features, ds_train_lfr.labels.ravel())

_ = evaluate_model(model, ds_test.features, ds_test.labels.ravel(), verbose=True)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 0.0

In [74]:
_ = calculate_utility_metrics(ds_test.labels.ravel(), [0] * len(ds_test.labels), verbose=True)

Metric          : Value          
Accuracy        : 0.759
Precision       : 0.380
Recall          : 0.500
F1              : 0.432


In [68]:
_ = evaluate_fairness(df_train_bias[target], np.array([0]*len(df_train_bias)), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   0.000
average_odds_difference         0.000
equal_opportunity_difference    0.000
disparate_impact                0.000
theil_index                     0.279


Multiple Runs cannot be done, because of using the StandardDataset by AIF360, which uses one-way one-hot encoding.

### Disparate Impact Remover
Lastly, we apply the DI-Remover to improve DI. For this, we start by reloading and converting the datasets.

In [76]:
df_train, df_test = load_data()
df_train = subsample(df_train)

In [77]:
df_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
df_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

Next, we apply the DI-Remover with different levels of repairment.

In [78]:
fairness_metrics = []
utility_metrics = []
for level in tqdm(np.linspace(0, 1, 10)):
    di = DisparateImpactRemover(repair_level=level)
    ds_train = di.fit_transform(df_train)
    ds_test = di.fit_transform(df_test)
    
    X_train, y_train = ds_train.features, ds_train.labels.ravel()
    X_test, y_test = ds_test.features, ds_test.labels.ravel()
    
    model = clone(clf)
    model.fit(X_train, y_train)

    utility_metrics.append(evaluate_model(model, X_test, y_test, verbose=False))
    fairness_metrics.append(evaluate_fairness(df_train_bias[target], model.predict(X_train), list(privileged_subset[0].keys()), verbose=False))


100%|██████████| 10/10 [01:07<00:00,  6.79s/it]


Lastly, we select the best performing result, based on DI and output fairness and utility metrics.

In [79]:
max_index, max_disparate_impact_row = max(enumerate(fairness_metrics), key=lambda x: x[1]['disparate_impact'])
max_disparate_impact_row

{'statistical_parity_difference': -0.886064512031419,
 'average_odds_difference': -0.7165976489885137,
 'equal_opportunity_difference': -0.7118014545283333,
 'disparate_impact': 0.03559399482142305,
 'theil_index': 0.07243322448451311}

In [81]:
utility_metrics[max_index]

{'Accuracy': 0.6137765234660814,
 'Precision': 0.6771994233452677,
 'Recall': 0.7314791687526085,
 'F1': 0.6045413669945015}

Multiple Runs cannot be done, because of using the StandardDataset by AIF360, which uses one-way one-hot encoding.